In [ ]:
# # Description
# 
# Prove that the Noise Reduction model works on a 2D polinomial.

# Add libs folder path to system variables to make the importation of local libraries easier.
# __________________________________________________________________________________________________
import os
import sys


CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)


# Import external and local libraries
# __________________________________________________________________________________________________
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import json
from typing import List

# Locals
from utils import *
from dataset import TensorDataset
from models import *
import config as cfg
from dlnr import DLNoiseReduction


# Global / Path variables
# __________________________________________________________________________________________________
DATA_PATH = os.path.join(CURRENT_DIR, 'data')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')


# Make sure that the GPU is being used
# __________________________________________________________________________________________________
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda"

In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
abalone = fetch_ucirepo(id=1) 
  
# data (as pandas dataframes) 
X = abalone.data.features 
y = abalone.data.targets

X = X.drop(columns=['Sex'], inplace=False)

df_data = pd.DataFrame(X, columns=abalone.data.feature_names)
df_data['target'] = y

scaler = MinMaxScaler()
df_data = pd.DataFrame(scaler.fit_transform(df_data), columns=df_data.columns)

X = df_data.drop(columns=['target'], inplace=False)
y = df_data['target']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Pre-denoising XAI Benchmarck

In [ ]:
model_params = {
    'ridge': {"alpha": 1.0},
    'pls': {"n_components": 1},
    'tree': {"max_depth": 5},
    'svm': {"dual": 'auto'},
    'knn': {"n_neighbors": 5, "weights": 'uniform', "algorithm": 'auto', "leaf_size": 30, "p": 2, "n_jobs": None},
}
xai_benchmark = XAI_benchmark(is_ts = False, model_params = model_params)

In [ ]:
xai_benchmark.fit(X_train, y_train.values.ravel())

In [ ]:
xai_benchmark.predict(X_test, y_test.values.ravel(), get_metrics=True)

In [ ]:
# Transform the data into a tensor
train_dataset = TensorDataset(
    x=X_train.values,
    y=y_train.values.reshape(-1, 1)
)
# Transform the data into a tensor
val_dataset = TensorDataset(
    x=X_test.values,
    y=y_test.values.reshape(-1, 1)
)

# Create the dataloaders
batch_size = 64
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

In [ ]:
model = GridFullyDenseNN(
    n_layers=4,
    hidden_layers=[
        (X_train.shape[1], 1024),
        (1024, 512),
        (512, 1024),
        (1024, 1),
    ],
    dropout_layers=[
        0.0,
        0.0,
        0.0,
        0.0,
    ],
    activation_func_layers=[
        nn.Sigmoid(),
        nn.Sigmoid(),
        nn.Sigmoid(),
        nn.Sigmoid(),
    ],
    want_dropout=[
        False,
        False,
        False,
        False,
    ],
    want_linear=[
        True,
        True,
        True,
        True,
    ],
    want_activation=[
        True,
        True,
        True,
        True,
    ],
)
model.to(device)

In [ ]:
lr = 0.001
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Define the trainer
trainer_basic = Trainer(
    model=model,
    train_generator=train_dataloader,
    val_generator=val_dataloader,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    epoch_scheduler=None,
    batch_scheduler=None,
    patience=15,
    epochs=500,
    checkpoints_path=os.path.join(CHECKPOINT_PATH, 'abalone_gradient')
)

Train the model

In [ ]:
type(y), y.shape

In [ ]:
model, train_losses, val_losses, best_train_loss, best_val_loss = trainer_basic.fit(verbose=True)

or load a checkpoint

In [ ]:
# model.load_state_dict(torch.load(os.path.join(CHECKPOINT_PATH, abalone_gradient.pth')))

In [ ]:
y_pred_test = model(torch.tensor(X_test.values).float().to(device)).cpu().detach().numpy().reshape(-1)

In [ ]:
# Show the metrics
gt_values = y_test.values
predicted_values = y_pred_test

mae = mean_absolute_error(gt_values, predicted_values)
mse = mean_squared_error(gt_values, predicted_values)
rmse = np.sqrt(mse)
# mape = np.mean(np.abs(gt_values - predicted_values) / gt_values) * 100
r_squared = r2_score(gt_values, predicted_values)
nn_benchmark = {
    'nn': {
        "MSE: ": mse,
        "RMSE: ": rmse,
        "MAE: ": mae,
        # "MAPE: ": mape,
        "R2: ": r_squared
    }
}
nn_benchmark

# Gradients to reduce noise

In [ ]:
X_no_noise = X.copy()
y_no_noise = y.copy()

In [ ]:
dlnr = DLNoiseReduction(model=model, criterion=criterion)
dlnr.fit(X.values, y.values.reshape(-1, 1))

In [ ]:
X_no_noise, y_no_noise = dlnr.transform(
                                        nrr=0.05,
                                        nr_threshold=0.01,
                                        max_epochs=200,
                                        plot_progress=False,
                                        path_to_save_imgs=None,
                                        denoise_y=True
)

# Denoised XAI Benchmark

In [ ]:
X_no_noise.shape, y_no_noise.shape

In [ ]:
xai_benchmark.fit(X_no_noise, y_no_noise.ravel())

In [ ]:
xai_benchmark.predict(X_no_noise, y_no_noise.ravel(), get_metrics=True)

In [ ]:
xai_benchmark.predict(X_test.values, y_test.values, get_metrics=True)